In [1]:
import time

from run_stacking import *

/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-19 11:03:27,718	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-12-19 11:03:27.927852: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = "ArrowHead"

X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(36, 1, 251) (36,) (175, 1, 251) (175,)


In [3]:
class TopNColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, n=2, proba_prefix="proba_model0_"):
        self.n = n
        self.proba_prefix = proba_prefix

    def fit(self, X: pl.DataFrame, y):
        # infer model names from column names
        self.models_ = np.unique(
            [
                c.replace(self.proba_prefix, "").split("_")[0]
                for c in X.columns
                if c.startswith(self.proba_prefix)
            ]
        ).tolist()

        self.classes_ = np.array(sorted(np.unique(y).tolist()))
        model_scores = []

        for model in self.models_:
            sel_cols = [c for c in X.columns if model in c]
            v = X.select(sel_cols)

            oof_pred_indices = np.argmax(v.to_numpy(), axis=1)
            oof_preds = self.classes_[oof_pred_indices]

            acc = accuracy_score(y, oof_preds)
            model_scores.append((acc, model))

        # sort by accuracy descending
        model_scores.sort(reverse=True)
        self.selected_models_ = [model for _, model in model_scores[: self.n]]

        return self

    def transform(self, X: pl.DataFrame) -> pl.DataFrame:
        selected_cols = []
        for model in self.selected_models_:
            selected_cols.extend([c for c in X.columns if model in c])

        return X.select(selected_cols)

In [4]:
class MeanTopN:
    def __init__(self, n=2):
        self.n = n
        # self.models = models

    def fit(self, X, y):
        self.proba_prefix = "proba_model0_"
        self.models = np.unique(
            [
                c.replace(self.proba_prefix, "").split("_")[0]
                for c in X.columns
                if c.startswith(self.proba_prefix)
            ]
        ).tolist()

        self.classes_ = np.array(sorted(np.unique(y).tolist()))
        self.model_scores = []
        for model in self.models:
            sel_cols = [c for c in X.columns if model in c]
            v = X.select(sel_cols)
            oof_pred_indices = np.argmax(v.to_numpy(), axis=1).tolist()
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            # print(v)
            # print(f"Model: {model}", oof_acc)
            self.model_scores.append((oof_acc, model))

    def predict_proba(self, probas):
        # select top n models
        self.model_scores.sort(reverse=True)
        selected_models = [model for score, model in self.model_scores[: self.n]]
        # print("Selected models:", selected_models)

        prob_list = []
        for model in selected_models:
            sel_cols = [c for c in probas.columns if model in c]
            v = probas.select(sel_cols)
            prob_list.append(v.to_numpy())

        # average probabilities
        avg_probas = np.mean(np.array(prob_list), axis=0)
        return avg_probas


class MeanMaxThreshold:
    def __init__(self, threshold=0.95):
        """
        threshold: float
            Only keep models whose accuracy is >= threshold * best_model_accuracy
        """
        self.threshold = threshold
        self.model_scores = []
        self.models = models
        self.selected_models = []

    def fit(self, X, y):
        self.proba_prefix = "proba_model0_"
        self.models = np.unique(
            [
                c.replace(self.proba_prefix, "").split("_")[0]
                for c in X.columns
                if c.startswith(self.proba_prefix)
            ]
        ).tolist()
        self.classes_ = np.array(sorted(np.unique(y).tolist()))
        self.model_scores = []

        for model in self.models:
            sel_cols = [c for c in X.columns if model in c]
            v = X.select(sel_cols)
            oof_pred_indices = np.argmax(v.to_numpy(), axis=1).tolist()
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            # print(f"Model: {model}, Accuracy: {oof_acc:.4f}")
            self.model_scores.append((oof_acc, model))

        # determine threshold
        best_acc = max(score for score, _ in self.model_scores)
        self.selected_models = [
            model for score, model in self.model_scores if score >= self.threshold * best_acc
        ]
        # print("Selected models within threshold:", self.selected_models)

    def predict_proba(self, probas):
        prob_list = []
        for model in self.selected_models:
            sel_cols = [c for c in probas.columns if model in c]
            v = probas[sel_cols]
            prob_list.append(v.to_numpy())

        # average probabilities across selected models
        avg_probas = np.mean(np.array(prob_list), axis=0)
        return avg_probas

In [ ]:
class StackerV4(BaseClassifier):
    def __init__(self, random_state=None, n_repetitions=1, k_folds=10, time_limit_in_minutes=0):
        super().__init__()
        self.n_repetitions = n_repetitions
        self.k_folds = k_folds
        self.random_state = random_state
        self.cv_splits = None
        self.time_limit_in_minutes = time_limit_in_minutes

    def get_model(self, name):
        if name == "multirocket-ridgecv":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={"hydra_": SparseScaler(), "multirocket_": StandardScaler()},
                    verbose=False,
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10)),
            )
            return pipe
        elif name == "all-ridgecv":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "rdst_": StandardScaler(),
                        #'quant_': RobustScaler(),
                        "hydra_": SparseScaler(),
                        "multirocket_": StandardScaler(),
                        #'catch22_': RobustScaler(),
                    },
                    verbose=False,
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10)),
            )
            return pipe
        elif name == "rstsf":
            return RSTSF(random_state=self.random_state, n_jobs=-1, n_estimators=100)
        elif name == "quant-etc":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "quant_": NoScaler(),
                    },
                    verbose=False,
                ),
                ExtraTreesClassifier(
                    n_estimators=200,
                    max_features=0.1,
                    criterion="entropy",
                    random_state=self.random_state,  # pass this in
                    n_jobs=-1,
                ),
            )
            return pipe
        elif name == "rdst-ridgecv":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "rdst_": StandardScaler(),
                    },
                    verbose=False,
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-4, 4, 20)),
            )
            return pipe
        elif name == "rdst-robustscale-ridgecv":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "rdst_": RobustScaler(),
                    },
                    verbose=False,
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-4, 4, 20)),
            )
            return pipe
        elif name == "catch22-quant-et":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "catch22_": NoScaler(),
                        "quant_": NoScaler(),
                    },
                    verbose=False,
                ),
                ExtraTreesClassifier(
                    n_estimators=200,
                    max_features=0.1,
                    criterion="entropy",
                    random_state=self.random_state,  # pass this in
                    n_jobs=-1,
                ),
            )
            return pipe
        elif name == "probability-linear-svc":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "proba_model0_": StandardScaler(),
                    },
                    verbose=False,
                ),
                SVC(kernel="linear", probability=True, random_state=self.random_state),
            )
            return pipe
        elif name == "probability-et":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "proba_model0_": StandardScaler(),
                    },
                    verbose=False,
                ),
                ExtraTreesClassifier(
                    n_estimators=500,
                    # max_features=0.3,
                    # criterion="entropy",
                    random_state=self.random_state,  # pass this in
                    n_jobs=-1,
                    bootstrap=True,
                ),
            )
            return pipe
        elif name == "probability-ridgecv":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "proba_model0_": StandardScaler(),
                    },
                    verbose=False,
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10)),
            )
            return pipe
        elif name == "probability-top2-ridgecv":
            pipe = make_pipeline(
                TopNColumnSelector(n=2),
                StandardScaler(),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10)),
            )
            return pipe
        # elif name == 'probability-rf':
        #    pipe = make_pipeline(
        #        MultiScaler(
        #            scalers={
        #                'proba_': StandardScaler(),
        #            },
        #            verbose=False
        #        ),
        #        RandomForestClassifier(n_estimators=200, random_state=self.random_state, n_jobs=-1)
        #    )
        #    return pipe
        elif name == "mean-treshold-90":
            return MeanMaxThreshold(threshold=0.90)
        elif name == "mean-treshold-99":
            return MeanMaxThreshold(threshold=0.99)
        elif name == "probability2-ridgecv":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "proba_model1_": StandardScaler(),
                    },
                    verbose=False,
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10)),
            )
            return pipe
        elif name == "probability2-et":
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        "proba_model1_": StandardScaler(),
                    },
                    verbose=False,
                ),
                ExtraTreesClassifier(
                    n_estimators=500,
                    # max_features=0.3,
                    # criterion="entropy",
                    random_state=self.random_state,  # pass this in
                    n_jobs=-1,
                    bootstrap=True,
                ),
            )
            return pipe
        else:
            raise ValueError(f"Unknown model name: {name}")

    def fit_tranformers(self, X):
        self.t1 = RandomDilatedShapeletTransform(n_jobs=-1, random_state=self.random_state)
        self.t2 = QUANTTransformer()
        self.t3 = MultiRocket(n_jobs=-1, random_state=self.random_state)
        self.t4 = HydraTransformer(n_jobs=-1, random_state=self.random_state)
        # self.t5 = Catch22(n_jobs=-1)
        self.t1.fit(X)
        self.t2.fit(X)
        self.t3.fit(X)
        self.t4.fit(X)
        # self.t5.fit(X)

    def transform_series(self, X):
        start = perf_counter()
        Xt1 = self.t1.transform(X)
        t1_time = perf_counter() - start
        print(f"RDST transform: {t1_time:.4f}s")

        start = perf_counter()
        Xt2 = self.t2.transform(X)
        t2_time = perf_counter() - start
        print(f"QUANT transform: {t2_time:.4f}s")

        start = perf_counter()
        Xt3 = self.t3.transform(X)
        t3_time = perf_counter() - start
        print(f"MultiRocket transform: {t3_time:.4f}s")

        start = perf_counter()
        Xt4 = self.t4.transform(X)
        t4_time = perf_counter() - start
        print(f"Hydra transform: {t4_time:.4f}s")

        return pl.DataFrame(
            np.hstack([Xt1, Xt2, Xt3, Xt4]),
            schema=[f"rdst_{i}" for i in range(Xt1.shape[1])]
            + [f"quant_{i}" for i in range(Xt2.shape[1])]
            + [f"multirocket_{i}" for i in range(Xt3.shape[1])]
            + [f"hydra_{i}" for i in range(Xt4.shape[1])],
            # [f'catch22_{i}' for i in range(Xt5.shape[1])]
        )

    def get_Xt(self):
        df = (
            pl.DataFrame(self.oof_predictions)
            .pivot(
                values="probability",
                index="index",
                on=["level", "model", "class"],
                aggregate_function="mean",
            )
            .sort("index")
        )

        rename_dict = {}
        for c in df.columns[1:]:
            t = tuple([v.replace("{", "").replace("}", "").replace('"', "") for v in c.split(",")])
            level, model_name, prob_class = t
            rename_dict[c] = f"proba_model{level}_{model_name}_class_{prob_class}"

        df = df.rename(rename_dict)  # .drop('index')

        return (
            self.Xt_.with_row_index("index")
            .join(df, on="index", how="full")
            .sort("index")
            .drop("index", "index_right")
        )

    def _fit(self, X, y):
        time_limit = self.time_limit_in_minutes * 60
        start_time = time.time()

        if self.cv_splits is None:
            self.cv_splits = generate_folds(
                X,
                y,
                n_splits=self.k_folds,
                n_repetitions=self.n_repetitions,
                random_state=self.random_state,
            )
        self.fit_tranformers(X)
        self.Xt_ = self.transform_series(X)
        self.trained_models_ = []

        self.tsc_algos = ["rstsf"]
        self.feature_algos = ["multirocket-ridgecv", "quant-etc", "rdst-ridgecv"]
        self.stacking_models = [
            "probability-ridgecv"
        ]  # , 'probability-et', 'probability-top2-ridgecv', 'mean-treshold-90']
        self.stacking_models_2 = []  #'probability2-ridgecv', 'probability2-et']

        self.oof_predictions = []

        print("-------------------------------------------------------")
        for model_name in self.tsc_algos:
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(X[train_idx], y[train_idx])
                proba = pipe.predict_proba(X[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    for scls, prob in zip(pipe.classes_, p):
                        d_ = {"index": idx, "model": model_name, "level": 0}
                        d_["class"] = scls.item()
                        d_["probability"] = prob.item()
                        self.oof_predictions.append(d_)

                model_group.append(pipe)
                print(f"Time elapsed: {time.time() - start_time:.2f}s / {time_limit:.2f}s")
            self.trained_models_.append((model_name, model_group))

            # for each model compute oof accuracy
            Xt = self.get_Xt()
            prob_columns = [col for col in Xt.columns if model_name in col]
            agg_probs = self.get_Xt().select(prob_columns)
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        print("-------------------------------------------------------")
        Xt = self.get_Xt()
        for model_name in self.feature_algos:
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(Xt[train_idx], y[train_idx])
                proba = pipe.predict_proba(Xt[val_idx])

                for idx, p in zip(val_idx, proba):
                    for scls, prob in zip(pipe.classes_, p):
                        d_ = {"index": idx, "model": model_name, "level": 0}
                        d_["class"] = scls.item()
                        d_["probability"] = prob.item()
                        self.oof_predictions.append(d_)

                model_group.append(pipe)
                print(f"Time elapsed: {time.time() - start_time:.2f}s / {time_limit:.2f}s")
            self.trained_models_.append((model_name, model_group))

            # for each model compute oof accuracy
            Xt = self.get_Xt()
            prob_columns = [col for col in Xt.columns if model_name in col]
            agg_probs = self.get_Xt().select(prob_columns)
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)

            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        print("-------------------------------------------------------")
        Xt = self.get_Xt()
        for model_name in self.stacking_models:
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(Xt[train_idx], y[train_idx])
                proba = pipe.predict_proba(Xt[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    for scls, prob in zip(pipe.classes_, p):
                        d_ = {"index": idx, "model": model_name, "level": 1}
                        d_["class"] = scls.item()
                        d_["probability"] = prob.item()
                        self.oof_predictions.append(d_)

                model_group.append(pipe)
                print(f"Time elapsed: {time.time() - start_time:.2f}s / {time_limit:.2f}s")

            self.trained_models_.append((model_name, model_group))

            Xt = self.get_Xt()
            prob_columns = [col for col in Xt.columns if model_name in col]
            agg_probs = self.get_Xt().select(prob_columns)
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)

            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        print("-------------------------------------------------------")
        for model_name in self.stacking_models_2:
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(self.Xt_[train_idx], y[train_idx])
                proba = pipe.predict_proba(self.Xt_[val_idx])

                for idx, p in zip(val_idx, proba):
                    for scls, prob in zip(pipe.classes_, p):
                        d_ = {"index": idx, "model": model_name, "level": 2}
                        d_["class"] = scls.item()
                        d_["probability"] = prob.item()
                        self.oof_predictions.append(d_)

                model_group.append(pipe)
                print(f"Time elapsed: {time.time() - start_time:.2f}s / {time_limit:.2f}s")

            self.trained_models_.append((model_name, model_group))

            Xt = self.get_Xt()
            prob_columns = [col for col in Xt.columns if model_name in col]
            agg_probs = self.get_Xt().select(prob_columns)
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)

            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        print("-------------------------------------------------------")

        stats_df = []
        for model in (
            self.tsc_algos + self.feature_algos + self.stacking_models + self.stacking_models_2
        ):
            stack = None
            if model in self.tsc_algos + self.feature_algos:
                stack = "0"
            elif model in self.stacking_models:
                stack = "1"
            elif model in self.stacking_models_2:
                stack = "2"
            else:
                raise ValueError(f"Unknown model {model} for stacking level")

            Xt_ = self.get_Xt()
            model_columns = [
                col for col in Xt_.columns if col.startswith(f"proba_model{stack}_{model}_class_")
            ]
            # print(model_columns)
            oof_proba = Xt_.select(model_columns)
            # print(oof_proba)
            # print(Xt_)
            oof_pred_indices = np.argmax(oof_proba, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            # print(f"{model}:{oof_acc:.4f}")
            stats_df.append(
                {
                    "model": model,
                    "stack": stack,
                    "oof_acc": oof_acc,
                }
            )

        stats_df = (
            pl.DataFrame(stats_df)
            .with_columns(
                pl.when(pl.col("model") == "probability-ridgecv")
                .then(pl.lit(1))
                .otherwise(pl.lit(0))
                .alias("is_preferred")
            )
            .filter(pl.col("stack") == "1")
            .sort(["oof_acc", "stack", "is_preferred"], descending=True)
        )
        # print(stats_df)

        self.best_model = (stats_df.row(0, named=True))["model"]
        print(f"Best model selected for prediction: {self.best_model}")

    def predict_proba_per_model(self, X):
        Xt = self.transform_series(X)
        return_dict = {}
        for model_name, model_group in self.trained_models_:
            oof_proba = []
            for model in model_group:
                if model_name in self.tsc_algos:
                    proba = model.predict_proba(X)
                else:
                    proba = model.predict_proba(Xt)
                prob_columns = []
                for i, p in enumerate(proba):
                    d = {
                        "index": i,
                    }
                    for scls, prob in zip(model.classes_, p):
                        stack = None
                        if model_name in self.tsc_algos + self.feature_algos:
                            stack = "0"
                        elif model_name in self.stacking_models:
                            stack = "1"
                        elif model_name in self.stacking_models_2:
                            stack = "2"
                        else:
                            raise ValueError(f"Unknown model {model} for stacking level")
                        k = f"proba_model{stack}_{model_name}_class_{scls}"
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
            df = pl.DataFrame(oof_proba).group_by("index").mean().sort("index").select(prob_columns)
            Xt = pl.concat([Xt, df], how="horizontal")
            return_dict[model_name] = df.to_numpy()
        return return_dict

    def _predict_proba(self, X):
        return self.predict_proba_per_model(X)[self.best_model]

    def _predict(self, X):
        probas = self._predict_proba(X)
        predicted_indices = np.argmax(probas, axis=1)
        return self.classes_[predicted_indices]

In [6]:
random_state = 8913029
n_repetitions = 2
kfolds = 3
s1 = Stacker(random_state=random_state, n_repetitions=n_repetitions)
s2 = StackerV4(random_state=random_state, n_repetitions=n_repetitions, k_folds=kfolds)

In [7]:
# from aeon.benchmarking import resampling # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# X_train, y_train, X_test, y_test = resampling.stratified_resample_data(X_train, y_train, X_test, y_test, random_state=0)

In [8]:
s2.fit(X_train, y_train)
pred = s2.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

RDST transform: 0.1652s
QUANT transform: 0.0828s
MultiRocket transform: 0.0735s
Hydra transform: 0.4245s
-------------------------------------------------------


 17%|█▋        | 1/6 [00:00<00:04,  1.23it/s]

Time elapsed: 2.08s / 0.00s


 33%|███▎      | 2/6 [00:01<00:03,  1.19it/s]

Time elapsed: 2.94s / 0.00s


...

Time elapsed: 26.17s / 0.00s
Time elapsed: 26.37s / 0.00s


100%|██████████| 6/6 [00:01<00:00,  4.47it/s]

Time elapsed: 26.57s / 0.00s


Model probability-ridgecv| CA: 0.7778
-------------------------------------------------------
-------------------------------------------------------
Best model selected for prediction: probability-ridgecv
RDST transform: 1.2444s
QUANT transform: 1.9858s
MultiRocket transform: 0.9118s
Hydra transform: 2.8783s


0.8628571428571429

In [9]:
s2.get_Xt()

rdst_0,rdst_1,rdst_2,rdst_3,rdst_4,rdst_5,rdst_6,rdst_7,rdst_8,rdst_9,rdst_10,rdst_11,rdst_12,rdst_13,rdst_14,rdst_15,rdst_16,rdst_17,rdst_18,rdst_19,rdst_20,rdst_21,rdst_22,rdst_23,rdst_24,rdst_25,rdst_26,rdst_27,rdst_28,rdst_29,rdst_30,rdst_31,rdst_32,rdst_33,rdst_34,rdst_35,rdst_36,…,hydra_5098,hydra_5099,hydra_5100,hydra_5101,hydra_5102,hydra_5103,hydra_5104,hydra_5105,hydra_5106,hydra_5107,hydra_5108,hydra_5109,hydra_5110,hydra_5111,hydra_5112,hydra_5113,hydra_5114,hydra_5115,hydra_5116,hydra_5117,hydra_5118,hydra_5119,proba_model0_rstsf_class_0,proba_model0_rstsf_class_1,proba_model0_rstsf_class_2,proba_model0_multirocket-ridgecv_class_0,proba_model0_multirocket-ridgecv_class_1,proba_model0_multirocket-ridgecv_class_2,proba_model0_quant-etc_class_0,proba_model0_quant-etc_class_1,proba_model0_quant-etc_class_2,proba_model0_rdst-ridgecv_class_0,proba_model0_rdst-ridgecv_class_1,proba_model0_rdst-ridgecv_class_2,proba_model1_probability-ridgecv_class_0,proba_model1_probability-ridgecv_class_1,proba_model1_probability-ridgecv_class_2
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.341647,223.0,18.0,0.670769,38.0,6.0,4.374766,43.0,34.0,0.930371,201.0,37.0,3.484193,103.0,44.0,2.109241,13.0,25.0,2.644291,29.0,0.0,1.608865,135.0,20.0,1.0020e-14,82.0,27.0,3.345007,40.0,20.0,3.038823,11.0,3.0,5.276702,0.0,0.0,1.958765,…,63.0,30.0,34.0,31.0,14.0,49.0,12.0,39.0,48.0,65.0,15.0,18.0,17.0,36.0,29.0,19.0,14.0,32.0,28.0,9.0,80.0,39.0,0.6,0.13,0.27,1.0,0.0,0.0,0.5375,0.1875,0.275,1.0,0.0,0.0,1.0,0.0,0.0
0.36182,222.0,37.0,0.387058,149.0,22.0,0.881683,53.0,34.0,1.377611,200.0,42.0,0.907458,98.0,37.0,1.640021,13.0,25.0,0.718113,29.0,5.0,1.506817,133.0,30.0,1.529881,96.0,27.0,4.319424,178.0,16.0,2.242498,11.0,14.0,3.403164,0.0,9.0,1.185482,…,61.0,20.0,15.0,30.0,15.0,77.0,5.0,43.0,66.0,69.0,3.0,12.0,12.0,40.0,31.0,8.0,3.0,36.0,32.0,4.0,95.0,41.0,0.01,0.77,0.22,0.0,1.0,0.0,0.0,0.665,0.335,0.0,1.0,0.0,0.0,1.0,0.0
0.2317,223.0,55.0,0.541048,23.0,17.0,2.264307,54.0,35.0,0.809481,200.0,43.0,0.940676,100.0,41.0,1.601888,9.0,24.0,1.539589,29.0,0.0,0.961335,135.0,22.0,1.416494,102.0,45.0,3.427789,35.0,16.0,0.668761,11.0,12.0,3.947306,0.0,5.0,1.99236,…,69.0,18.0,25.0,40.0,11.0,63.0,5.0,48.0,66.0,78.0,0.0,7.0,14.0,32.0,32.0,10.0,9.0,29.0,29.0,8.0,91.0,42.0,0.07,0.325,0.605,0.0,0.0,1.0,0.05,0.2,0.75,0.0,0.0,1.0,0.0,0.0,1.0
0.354914,224.0,18.0,0.356866,17.0,3.0,6.311697,44.0,11.0,1.184559,199.0,34.0,5.528756,111.0,17.0,1.428048,11.0,24.0,4.804073,28.0,0.0,2.979737,137.0,14.0,1.408885,74.0,24.0,2.248571,36.0,25.0,4.680013,14.0,0.0,6.705574,0.0,0.0,3.875619,…,57.0,23.0,33.0,33.0,16.0,46.0,23.0,31.0,43.0,52.0,25.0,24.0,18.0,34.0,30.0,27.0,16.0,29.0,23.0,15.0,73.0,37.0,0.935,0.01,0.055,1.0,0.0,0.0,0.975,0.0025,0.0225,1.0,0.0,0.0,1.0,0.0,0.0
3.3723e-14,224.0,36.0,0.583731,25.0,21.0,2.002461,51.0,33.0,0.953651,196.0,41.0,1.392861,97.0,39.0,1.196643,13.0,24.0,1.415668,29.0,4.0,1.173174,130.0,28.0,1.510426,92.0,37.0,3.25459,41.0,21.0,2.2428,9.0,10.0,3.822072,0.0,8.0,0.945539,…,63.0,24.0,22.0,30.0,17.0,57.0,10.0,49.0,52.0,65.0,10.0,13.0,16.0,35.0,35.0,10.0,1.0,41.0,35.0,7.0,73.0,48.0,0.06,0.7,0.24,0.0,1.0,0.0,0.06,0.7725,0.1675,0.0,1.0,0.0,0.0,1.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.317693,222.0,29.0,0.345659,25.0,26.0,0.0,53.0,34.0,1.786699,200.0,41.0,1.778293,99.0,37.0,2.171006,12.0,25.0,0.450733,29.0,5.0,2.156807,131.0,29.0,1.917477,95.0,26.0,4.398144,52.0,13.0,2.768766,13.0,11.0,2.999339,0.0,9.0,1.229246,…,56.0,24.0,26.0,28.0,13.0,69.0,11.0,39.0,59.0,65.0,12.0,16.0,13.0,35.0,28.0,12.0,9.0,33.0,34.0,8.0,95.0,31.0,0.04,0.78,0.18,0.0,1.0,0.0,0.035,0.685,0.28,0.0,1.0,0.0,0.0,1.0,0.0
0.369485,221.0,

In [11]:
P = s2.predict_proba_per_model(X_test).items()
print()
for model, prov_pred in P:
    oof_pred_indices = np.argmax(prov_pred, axis=1)
    oof_preds = s2.classes_[oof_pred_indices]
    oof_acc = accuracy_score(y_test, oof_preds)
    print(f"Model {model}| CA: {oof_acc:.4f}")

RDST transform: 0.6789s
QUANT transform: 0.2895s
MultiRocket transform: 0.1438s
Hydra transform: 0.5585s

Model rstsf| CA: 0.7429
Model multirocket-ridgecv| CA: 0.8800
Model quant-etc| CA: 0.8000
Model rdst-ridgecv| CA: 0.8743
Model probability-ridgecv| CA: 0.8629


In [12]:
s2.Xt_.select([c for c in s2.Xt_.columns if c.startswith("proba_model1")])

shape: (0, 0)
┌┐
╞╡
└┘

In [10]:
dsfgDF = gdfg

NameError: name 'gdfg' is not defined

In [13]:
m = MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

0.8742857142857143

In [14]:
from aeon.classification.interval_based import RSTSF

m = RSTSF(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

0.7542857142857143

In [ ]:
from aeon.classification.hybrid import HIVECOTEV2

m = HIVECOTEV2(time_limit_in_minutes=3, random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.interval_based import DrCIFClassifier

m = DrCIFClassifier(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.dictionary_based import TemporalDictionaryEnsemble

m = TemporalDictionaryEnsemble(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.shapelet_based import ShapeletTransformClassifier

m = ShapeletTransformClassifier(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
X_train.shape

In [ ]:
from aeon.transformations.collection.shapelet_based import SAST

sast = SAST(n_jobs=-1)
sast.fit(X_train, y_train)

sast.transform(X_train)

In [ ]:
fsdf = dfgdgf

In [ ]:
s1.fit(X_train, y_train)
pred = s1.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
import numpy as np

# Generate random data
np.random.seed(42)
X_train = np.random.randn(100, 20)  # 100 samples, 20 features
y_train = np.random.randint(0, 3, 100)  # 3 classes

X_test = np.random.randn(30, 20)
y_test = np.random.randint(0, 3, 30)

# Train RidgeClassifierCV
model = RidgeClassifierCVIndicator(alphas=[0.1, 1.0, 10.0, 100.0])
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)
score = model.score(X_test, y_test)

print(f"Best alpha: {model.alpha_}")
print(f"Test accuracy: {score:.3f}")
print(f"Predictions: {predictions[:10]}")

In [ ]:
model.predict_proba(X_test)

In [ ]:
model.predict(X_test)